# Lezione 07 — Servire un modello

**Corso:** MLOps e Monitoraggio con Python · **Blocco B**
**Tempo stimato:** 110 minuti · **Capitolo del manuale:** `manual/07_servir_modelo_IT.md`

## Cosa vedrai in questa lezione

- Riaddestrare la `Pipeline` vincitrice (regressione logistica, `class_weight="balanced"`, `C=1.0`) e salvarla come artefatto su disco (`modelo.pkl`), esattamente come la caricherebbe un servizio reale.
- Validare l'input con un modello **Pydantic** costruito a partire dalle colonne di `churn.csv`.
- Costruire un'API con **FastAPI** con due endpoint: uno **online** (`/predecir`, un cliente) e uno **a lotti** (`/predecir_lote`, una lista di clienti).
- Testarla con `fastapi.testclient.TestClient`, senza aprire nessuna porta né eseguire `uvicorn.run` in background.
- Misurare la **latenza reale** con `time.perf_counter()`, su più richieste, con media e mediana.
- Esportare il modello in **ONNX** con `skl2onnx` e confrontare la sua latenza con quella del modello originale, con lo stesso input e lo stesso numero di chiamate.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook.

## Prerequisiti

- Dalle lezioni 4 e 5: la configurazione vincitrice del modello di churn (regressione logistica, `class_weight="balanced"`, `C=1.0`, F1≈0,3166 sul set di test). Questa lezione la riaddestra da zero, non dipende da nessun file salvato nelle lezioni precedenti.
- La cartella `datos/` del corso, con `churn.csv` dentro, e aver letto `datos/LEEME.md`.
- Python di base: funzioni, dizionari, liste per comprensione.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca (incluso FastAPI, skl2onnx e onnxruntime, che potrebbero non essere preinstallati).

In [ ]:
import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
asegurar("skl2onnx", "skl2onnx")
asegurar("onnx", "onnx")
asegurar("onnxruntime", "onnxruntime")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, definiamo `cargar_datos`, `SEMILLA = 42` e le colonne di input.

In [ ]:
import os
import shutil
import time
import json
import warnings

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# Silenziamo un avviso di compatibilità di starlette/httpx (fisso, non aggiunge nulla qui) perché
# l'output di questo notebook non dipenda da dettagli interni di versioni delle librerie
warnings.filterwarnings("ignore", category=DeprecationWarning)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerie e funzioni ausiliarie pronte.")

---
## Cartella di lavoro effimera

Come nelle lezioni precedenti: creiamo `proyecto_churn/` da zero.

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## 1. Da un modello in un notebook a un modello servito

Fino alla lezione 6, "usare il modello" significava chiamare `pipeline.predict(...)` dentro lo stesso processo Python che lo aveva addestrato. Funziona per sperimentare, ma un sistema reale quasi mai ha questa fortuna: chi ha bisogno di una previsione (un'applicazione web, un altro servizio, un processo batch notturno) di solito viene eseguito in **un altro processo**, a volte su **un'altra macchina**, scritto a volte in **un altro linguaggio**. Il modo standard per risolverlo è avvolgere il modello in un'**API**: un programma che riceve richieste via HTTP, le valida, chiama il modello e restituisce una risposta, senza che chi chiama debba sapere nulla di scikit-learn.

> **Nota:** questo notebook testa l'API con `fastapi.testclient.TestClient`, che chiama l'applicazione ASGI direttamente in memoria. Non apre nessuna porta né socket reale, e per questo il risultato è deterministico e non dipende da quali altri processi stiano usando la rete del tuo computer — il modo corretto di testare FastAPI dentro un notebook o una suite di test (a differenza di eseguire `uvicorn.run` in background, che aprirebbe davvero una porta).

---
## 2. Riaddestrare la configurazione vincitrice e salvarla come artefatto

Riaddestriamo esattamente la configurazione vincitrice delle lezioni 4 e 5 (regressione logistica, `class_weight="balanced"`, `C=1.0`, partizione 75/25 con seme 42).

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

# Stessa configurazione vincitrice delle lezioni 4 e 5: regressione logistica, class_weight="balanced", C=1.0
preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
f1 = round(f1_score(y_test, y_pred), 4)
auc = round(roc_auc_score(y_test, y_proba), 4)
print("F1 (set di test):", f1)
print("AUC (set di test):", auc)
assert f1 > 0.25 and auc > 0.6, "il ri-addestramento non ha riprodotto il modello atteso"

Salviamo la `Pipeline` addestrata su disco con `joblib.dump`: quel file (`modelo.pkl`) è l'**artefatto** che un'API reale caricherebbe all'avvio, senza riaddestrare nulla. Per verificare che l'artefatto sia fedele, lo ricarichiamo in una variabile distinta (`MODELO`, in maiuscolo, per distinguerla dalla `pipeline` appena addestrata) e verifichiamo che predica esattamente uguale.

In [ ]:
# Salviamo la Pipeline addestrata come un file, esattamente l'artefatto che l'API caricherà
RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)
tamano_kb = round(os.path.getsize(RUTA_MODELO) / 1024, 1)
print("Artefatto salvato in:", RUTA_MODELO, f"(~{tamano_kb} KB)")

# L'API non importa mai "pipeline" da questo notebook: carica il file, come farebbe un processo diverso
MODELO = joblib.load(RUTA_MODELO)
assert np.array_equal(MODELO.predict(X_test), pipeline.predict(X_test))
print("Il modello ricaricato da disco predice esattamente come l'originale.")

---
## 3. Validare l'input con Pydantic

Prima di costruire l'API definiamo un modello Pydantic, `Cliente`, con esattamente le colonne di `churn.csv` che la Pipeline si aspetta come input (tutte tranne `abandono`, che è l'etichetta da predire). Prima verifichiamo i valori reali di ogni colonna categorica, per non inventare categorie che non esistono nei dati.

In [ ]:
# I valori ammessi di ogni colonna categorica, presi dai dati reali (non inventati)
print("valori di 'plan':", sorted(df["plan"].unique()))
print("valori di 'contrato':", sorted(df["contrato"].unique()))
print("valori di 'metodo_pago':", sorted(df["metodo_pago"].unique()))

`Literal[...]` in Pydantic rifiuta qualsiasi valore che non sia nella lista, con un errore di validazione chiaro, prima che quel dato arrivi anche solo a toccare il modello. `Field(ge=0)` richiede che un numero sia maggiore o uguale a zero (non può esserci un'anzianità negativa). Il campo `id_cliente` non partecipa alla previsione: viaggia con la richiesta solo per poter identificare a quale cliente appartiene ogni risposta.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Cliente(BaseModel):
    """Un modello Pydantic con esattamente le colonne di input di churn.csv (tutte tranne 'abandono' e 'id_cliente', conservata per identificare la risposta)."""
    id_cliente: str
    antiguedad_meses: int = Field(ge=0, description="mesi come cliente, non può essere negativo")
    plan: Literal["basico", "estandar", "premium"]
    contrato: Literal["mensual", "anual"]
    uso_mensual_horas: float = Field(ge=0)
    tickets_soporte: int = Field(ge=0)
    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]
    factura_mensual: float = Field(ge=0)


def cliente_a_fila(cliente):
    """Converte un Cliente validato nella riga con una colonna per variabile che la Pipeline si aspetta."""
    datos = cliente.model_dump()
    return pd.DataFrame([datos])[COLUMNAS_ENTRADA]

print("Modello Pydantic 'Cliente' definito, con le", len(COLUMNAS_ENTRADA), "colonne di input.")

### 3.1 Testare la validazione con un caso buono e uno cattivo

In [ ]:
cliente_ok = {
    "id_cliente": "C_DEMO_0001", "antiguedad_meses": 12, "plan": "estandar", "contrato": "mensual",
    "uso_mensual_horas": 15.0, "tickets_soporte": 1, "metodo_pago": "tarjeta", "factura_mensual": 22.5,
}
c = Cliente(**cliente_ok)
print("Cliente valido:", c)

# Un valore di 'plan' che non esiste nei dati: Pydantic lo rifiuta PRIMA di arrivare al modello
cliente_malo = dict(cliente_ok, plan="diamante")
try:
    Cliente(**cliente_malo)
    raise AssertionError("doveva lanciare un errore di validazione")
except Exception as e:
    print("Rifiutato come previsto. Tipo di errore:", type(e).__name__)

**Cosa osservi?** Il cliente con `plan="diamante"` non arriva mai a diventare una riga per il modello: Pydantic lancia un errore di validazione (`ValidationError`) nel momento di costruire l'oggetto `Cliente`, prima di qualsiasi calcolo. Questo è esattamente ciò che si cerca validando l'input: rifiutare dati scorretti il più vicino possibile all'origine, con un messaggio chiaro, invece di lasciare che arrivino al modello e produzcano un risultato silenziosamente senza senso.

---
## 4. Costruire l'API con FastAPI: online contro a lotti

Definiamo due endpoint con lo stesso modello caricato in memoria (`MODELO`): `/predecir`, che riceve UN `Cliente` e risponde con UNA previsione (servizio **online**), e `/predecir_lote`, che riceve una LISTA di `Cliente` e risponde con una LISTA di previsioni (servizio **a lotti**).

In [ ]:
from fastapi import FastAPI
from typing import List

app = FastAPI(title="API di churn (lezione 7)")


@app.post("/predecir")
def predecir(cliente: Cliente):
    """Servizio ONLINE: una richiesta, un cliente, risposta immediata."""
    fila = cliente_a_fila(cliente)
    probabilidad = float(MODELO.predict_proba(fila)[0, 1])
    return {
        "id_cliente": cliente.id_cliente,
        "probabilidad_abandono": round(probabilidad, 4),
        "abandono_previsto": int(probabilidad >= 0.5),
    }


@app.post("/predecir_lote")
def predecir_lote(clientes: List[Cliente]):
    """Servizio A LOTTI (batch): una lista di clienti, una lista di risposte, senza fretta."""
    filas = pd.DataFrame([c.model_dump() for c in clientes])[COLUMNAS_ENTRADA]
    probabilidades = MODELO.predict_proba(filas)[:, 1]
    return [
        {"id_cliente": c.id_cliente, "probabilidad_abandono": round(float(p), 4), "abandono_previsto": int(p >= 0.5)}
        for c, p in zip(clientes, probabilidades)
    ]


print("API definita, con due endpoint: /predecir (online) e /predecir_lote (a lotti).")

| | Online (`/predecir`) | A lotti (`/predecir_lote`) |
|---|---|---|
| Input | un `Cliente` | una lista di `Cliente` |
| Quando si usa | un'applicazione chiede una risposta per UN utente, ORA | un processo notturno ricalcola il rischio di TUTTO il portafoglio clienti |
| Limite di tempo | conta molto: l'utente sta aspettando | conta poco: può richiedere minuti |
| Costo per riga | maggiore (validazione e overhead della richiesta per ogni riga) | minore (l'overhead si distribuisce su molte righe, sezione 4.1) |

> **Regola pratica:** se la risposta la aspetta una persona dall'altra parte, sul momento, è un servizio online e la latenza è un vincolo reale di prodotto. Se la risposta alimenta un processo che si esegue senza nessuno in attesa (un report, una tabella che si aggiorna ogni notte), è un servizio a lotti e ciò che conta è il **throughput** (righe al secondo), non la latenza di una riga.

---
## 5. Testare l'API con `TestClient`

Testiamo prima il caso normale (una riga reale del set di test) e verifichiamo la struttura esatta della risposta.

In [ ]:
from fastapi.testclient import TestClient

# TestClient chiama l'applicazione ASGI direttamente in memoria: NON apre nessuna porta né socket reale,
# per questo è il modo corretto di testare un'API FastAPI in un notebook, senza uvicorn.run in background
cliente_api = TestClient(app)

fila_prueba = X_test.iloc[0].to_dict()
fila_prueba["id_cliente"] = "C_TEST_0001"

respuesta = cliente_api.post("/predecir", json=fila_prueba)
print("codice di stato:", respuesta.status_code)
print("corpo della risposta:", respuesta.json())
assert respuesta.status_code == 200
assert set(respuesta.json().keys()) == {"id_cliente", "probabilidad_abandono", "abandono_previsto"}

### 5.1 Testare i casi di input non valido

In [ ]:
fila_invalida = dict(fila_prueba, plan="diamante")
respuesta_mala = cliente_api.post("/predecir", json=fila_invalida)
print("codice di stato con plan non valido:", respuesta_mala.status_code)
assert respuesta_mala.status_code == 422, "Pydantic doveva rifiutare 'diamante' come plan"

fila_negativa = dict(fila_prueba, antiguedad_meses=-3)
respuesta_negativa = cliente_api.post("/predecir", json=fila_negativa)
print("codice di stato con antiguedad_meses negativa:", respuesta_negativa.status_code)
assert respuesta_negativa.status_code == 422

**Cosa osservi?** FastAPI traduce automaticamente qualsiasi errore di validazione di Pydantic in una risposta HTTP **422 Unprocessable Entity**, senza che abbiamo scritto nessuna riga di codice per questo: la validazione della sezione 3 e il routing di FastAPI lavorano insieme. Chi chiama l'API riceve un codice di errore chiaro, non un'eccezione interna di scikit-learn né (peggio) una previsione calcolata su dati corrotti.

---
## 6. Misurare la latenza reale del servizio online

Misuriamo il tempo di risposta di `/predecir` con `time.perf_counter()`, ripetendo la stessa richiesta 200 volte (dopo un piccolo "riscaldamento"), e riportiamo media e mediana -- non un singolo tempo isolato, che può essere un caso.

In [ ]:
def medir_latencias_ms(funcion_llamada, n_peticiones):
    """Esegue funcion_llamada() n_peticiones volte e restituisce la lista dei tempi in millisecondi."""
    tiempos = []
    for _ in range(n_peticiones):
        inicio = time.perf_counter()
        funcion_llamada()
        tiempos.append((time.perf_counter() - inicio) * 1000)
    return np.array(tiempos)


N_PETICIONES = 200
# Riscaldiamo con poche chiamate prima di misurare (la prima chiamata include sempre costi di inizializzazione)
for _ in range(10):
    cliente_api.post("/predecir", json=fila_prueba)

latencias_online = medir_latencias_ms(lambda: cliente_api.post("/predecir", json=fila_prueba), N_PETICIONES)
media_online = round(float(latencias_online.mean()), 3)
mediana_online = round(float(np.median(latencias_online)), 3)
print("Latenza dell'endpoint /predecir su", N_PETICIONES, "richieste -- media:", media_online, "ms, mediana:", mediana_online, "ms")
print("(CPU locale, esempio illustrativo: include la validazione di Pydantic e lo strato ASGI del TestClient; i tempi assoluti variano tra macchine.)")
assert media_online < 500, "una singola richiesta non dovrebbe richiedere mezzo secondo in un'API così piccola"

### 6.1 A lotti contro online: l'overhead si amortizza

In [ ]:
lote_100 = X_test.head(100).to_dict("records")
for i, fila in enumerate(lote_100):
    fila["id_cliente"] = f"C_LOTE_{i:04d}"

inicio = time.perf_counter()
respuesta_lote = cliente_api.post("/predecir_lote", json=lote_100)
tiempo_lote_ms = (time.perf_counter() - inicio) * 1000
print("codice di stato del lotto:", respuesta_lote.status_code)
assert respuesta_lote.status_code == 200
assert len(respuesta_lote.json()) == 100

tiempo_por_fila_lote = round(tiempo_lote_ms / 100, 4)
print("Lotto di 100 clienti in un'unica richiesta:", round(tiempo_lote_ms, 3), "ms in totale --", tiempo_por_fila_lote, "ms per riga")
print("Contro la mediana online (una riga per richiesta):", mediana_online, "ms")

**Cosa osservi?** Il tempo per riga dentro il lotto di 100 è molto minore della latenza di una richiesta online. La differenza non è che ONNX o il modello siano diversi: è che in una richiesta online, ogni riga paga SEPARATAMENTE il costo fisso della validazione di Pydantic e dello strato HTTP/ASGI, mentre nel lotto quel costo fisso si paga UNA VOLTA e si distribuisce sulle 100 righe. È la ragione concreta per cui un processo batch notturno raggruppa le righe invece di chiamare 100 volte `/predecir`.

---
## 7. Esportare il modello in ONNX

**ONNX** (Open Neural Network Exchange) è un formato aperto per rappresentare modelli in modo indipendente dal framework in cui sono stati addestrati. `skl2onnx` converte una `Pipeline` di scikit-learn (preprocessamento + modello) in un grafo ONNX che si può eseguire con `onnxruntime`, un motore di inferenza compilato e ottimizzato, senza dipendere da tutto il pacchetto scikit-learn nel servizio finale.

In [ ]:
from skl2onnx import to_onnx
from skl2onnx.common.data_types import StringTensorType, FloatTensorType

# skl2onnx ha bisogno di un tipo dichiarato per ogni colonna di input (qui vanno "colonna per colonna", come si aspetta il ColumnTransformer)
tipos_entrada = [(col, StringTensorType([None, 1])) for col in COLUMNAS_CATEGORICAS]
tipos_entrada += [(col, FloatTensorType([None, 1])) for col in COLUMNAS_NUMERICAS]

modelo_onnx = to_onnx(pipeline, initial_types=tipos_entrada, target_opset=17)
RUTA_ONNX = os.path.join(CARPETA, "modelo.onnx")
with open(RUTA_ONNX, "wb") as f:
    f.write(modelo_onnx.SerializeToString())
tamano_onnx_kb = round(os.path.getsize(RUTA_ONNX) / 1024, 1)
print("Modello esportato in ONNX in:", RUTA_ONNX, f"(~{tamano_onnx_kb} KB)")

### 7.1 Caricare il modello ONNX e verificare che predica lo stesso

In [ ]:
import onnxruntime as ort

sesion_onnx = ort.InferenceSession(RUTA_ONNX, providers=["CPUExecutionProvider"])
print("output del modello ONNX:", [o.name for o in sesion_onnx.get_outputs()])


def fila_a_entradas_onnx(fila):
    """Converte una riga (dict) nel dizionario di array che onnxruntime si aspetta, un input per colonna."""
    entradas = {col: np.array([[fila[col]]], dtype=object) for col in COLUMNAS_CATEGORICAS}
    entradas.update({col: np.array([[fila[col]]], dtype=np.float32) for col in COLUMNAS_NUMERICAS})
    return entradas


entradas_prueba = fila_a_entradas_onnx(fila_prueba)
salida_onnx = sesion_onnx.run(None, entradas_prueba)
probabilidad_onnx = float(salida_onnx[1][0][1])
probabilidad_sklearn = float(MODELO.predict_proba(cliente_a_fila(Cliente(**fila_prueba)))[0, 1])
print("probabilità scikit-learn:", round(probabilidad_sklearn, 6))
print("probabilità ONNX:", round(probabilidad_onnx, 6))
assert abs(probabilidad_sklearn - probabilidad_onnx) < 1e-4, "le due probabilità dovrebbero coincidere fino all'arrotondamento in virgola mobile"
print("Stesso risultato (fino all'arrotondamento in virgola mobile): l'esportazione non ha cambiato ciò che il modello predice.")

---
## 8. Confrontare la latenza: scikit-learn contro ONNX

Confrontiamo con lo STESSO input e lo STESSO numero di chiamate (200), ma chiamando direttamente ciascun modello -- senza il livello dell'API -- per isolare il costo puro dell'inferenza dal costo di validazione e HTTP già misurato nella sezione 6.

In [ ]:
# Confronto onesto: STESSO input (fila_prueba), STESSO numero di chiamate, senza il livello dell'API in nessuno dei due casi
fila_prueba_df = cliente_a_fila(Cliente(**fila_prueba))

for _ in range(10):
    pipeline.predict_proba(fila_prueba_df)
    sesion_onnx.run(None, entradas_prueba)

latencias_sklearn = medir_latencias_ms(lambda: pipeline.predict_proba(fila_prueba_df), N_PETICIONES)
latencias_onnx = medir_latencias_ms(lambda: sesion_onnx.run(None, entradas_prueba), N_PETICIONES)

media_sklearn = round(float(latencias_sklearn.mean()), 4)
mediana_sklearn = round(float(np.median(latencias_sklearn)), 4)
media_onnx = round(float(latencias_onnx.mean()), 4)
mediana_onnx = round(float(np.median(latencias_onnx)), 4)

print("scikit-learn (Pipeline.predict_proba),", N_PETICIONES, "chiamate -- media:", media_sklearn, "ms, mediana:", mediana_sklearn, "ms")
print("ONNX Runtime (InferenceSession.run),", N_PETICIONES, "chiamate -- media:", media_onnx, "ms, mediana:", mediana_onnx, "ms")
print("(CPU locale, esempio illustrativo: i tempi assoluti variano tra macchine ed esecuzioni; qui si ricalcola l'ordine di grandezza, non un millisecondo esatto.)")
assert media_sklearn > 0 and media_onnx > 0

**Cosa osservi?** Su questa macchina e per questo modello piccolo (una regressione logistica su poche colonne), ONNX Runtime risulta notevolmente più rapido che chiamare `predict_proba` sulla `Pipeline` di scikit-learn: la maggior parte del tempo che impiega scikit-learn non è il calcolo del modello in sé, ma l'overhead di ricostruire un `DataFrame` di pandas e di passare per il `ColumnTransformer` in Python puro a ogni chiamata, cosa che il grafo ONNX compilato evita. Questo **non è una legge generale**: per modelli molto più grandi, o con un preprocessamento diverso, la differenza può essere minore o persino sparire -- per questo il capitolo misura, invece di supporre, e per questo dichiariamo esplicitamente che la cifra è di una macchina locale specifica.

> **Attenzione:** il risultato onesto di questa sezione può variare da una macchina all'altra, e a volte anche da un'esecuzione all'altra sulla stessa macchina. Ciò che non dovrebbe variare è l'ordine di grandezza né la spiegazione di dove viene la differenza (overhead di Python/pandas contro un grafo compilato); se sulla tua macchina scikit-learn risulta più rapido o simile, resta un risultato valido -- la conclusione corretta è "misura nel tuo ambiente di produzione", non "ONNX vince sempre".

---
## 9. Pulizia finale della cartella di lavoro

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Riepilogo

- Abbiamo riaddestrato la configurazione vincitrice (regressione logistica, `class_weight="balanced"`, `C=1.0`) e l'abbiamo salvata come **artefatto** (`modelo.pkl`) con `joblib`, esattamente come la caricherebbe un servizio reale.
- Un modello **Pydantic** (`Cliente`) valida l'input con le colonne reali di `churn.csv`: rifiuta categorie inesistenti e valori negativi PRIMA che arrivino al modello, con un HTTP 422 automatico.
- **FastAPI** esporta due endpoint: `/predecir` (servizio **online**, un cliente, la latenza conta) e `/predecir_lote` (servizio **a lotti**, una lista, conta il throughput).
- `TestClient` testa l'API in memoria, senza aprire nessuna porta né eseguire `uvicorn.run`.
- La latenza reale (`time.perf_counter()`, 200 richieste) dell'endpoint online è dell'ordine di pochi millisecondi per richiesta su questa macchina; il costo per riga scende molto raggruppando in un lotto, perché l'overhead fisso si distribuisce.
- **ONNX** (`skl2onnx` + `onnxruntime`) è risultato considerevolmente più rapido di scikit-learn puro per questo modello piccolo, con lo stesso input e lo stesso numero di chiamate -- un risultato misurato, non supposto, e che potrebbe non ripetersi uguale su un'altra macchina o con un altro modello.

*Continua con la lezione 8: Contenitori e CI/CD.*